# Prompt budget, round 2: does any model lose a pass?

Round 2 (`__ML_PROMPT_VARIANT__="cuts2"`, `src/tools/tool-details.ts`) moves `locate`'s option texts and
`python_exec`'s `tables`/`cast`/`mode` texts out of every call and into `agent_api_docs({ tool })`, and makes the
"Incognito is off" rendering error lead with the retry that works. The ship rule (`docs/dev/prompt-review.md`): **no
model's pass rate drops.** Each task needs something that moved, and is scored from the run itself.

Sweeps read here:

- `prompt-budget-cuts2`: 11 models x 4 tasks x 2 prompts x 3 repeats (the first look).
- `cuts2-recheck2`: the three models that dropped there (MiniMax M3, qwen3.6, glm-4.7-flash), x 8 repeats, on main
  `1eb1448c` plus the variant.
- `cuts2-python-fix` (MiniMax) and `cuts2-python-fix-glm` (glm-4.7-flash): the two table tasks, x 8, after one
  sentence was added to the short `tables` text.

In [1]:
import json, os
from pathlib import Path
import pandas as pd
from scipy.stats import fisher_exact
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "bench"
SWEEP_ROOT = Path(os.environ.get("SWEEP_ROOT", Path.home() / "git/window-ml-cuts2/tests/e2e/artifacts/bench"))
SWEEPS = ["prompt-budget-cuts2", "cuts2-recheck2", "cuts2-python-fix", "cuts2-python-fix-glm"]
FIX_SWEEPS = ["cuts2-python-fix", "cuts2-python-fix-glm"]
EXTRACT = HERE / "data" / "prompt-cuts2" / "runs.json"
pd.set_option("display.width", 160)

## The runs, pinned

One row per run, read from each run's `cell.json` (outcome, tokens, answer) and `run.json` (its steps). The rows are
kept in `data/prompt-cuts2/runs.json`, so the numbers below do not depend on the sweep folders still existing.

In [2]:
def read_sweeps():
    rows = []
    for sweep in SWEEPS:
        for cell in sorted((SWEEP_ROOT / sweep).glob("*/*/r*/cell.json")):
            if "history" in cell.parts:
                continue
            c = json.loads(cell.read_text())
            m = c.get("measurement") or {}
            steps = json.loads((cell.parent / "run.json").read_text())["session"]["steps"] if (cell.parent / "run.json").exists() else []
            tools = [s.get("tool") for s in steps if s.get("tool")]
            py_code = [str((s.get("arguments") or {}).get("code", "")) for s in steps if s.get("tool") == "python_exec"]
            rows.append({
                "sweep": sweep, "task": c["taskId"], "prompt": c["combo"]["prompt"], "model": c["combo"]["model"],
                "repeat": c["repeat"], "passed": bool(m.get("succeeded")), "hit_cap": bool(m.get("hitCap")),
                "steps": m.get("steps"), "calls": sum(1 for s in steps if s.get("usage")),
                "prompt_tokens": (m.get("tokens") or {}).get("prompt"),
                "answer": (m.get("finalAnswer") or m.get("answer") or "")[:160],
                "tools": tools, "tables_in_code": any("tables=" in s for s in py_code),
            })
    return rows

rows = wmlnb.extract(EXTRACT, read_sweeps, source={"sweeps": SWEEPS, "sweep_root": "~/git/window-ml-cuts2/tests/e2e/artifacts/bench"})
runs = pd.DataFrame(rows)
pd.DataFrame(wmlnb.provenance(EXTRACT))

,extract,sha256,rows,extracted_at,commit,sweep_root,sweeps
0,notebooks/bench/data/prompt-cuts2/runs.json,812ac591508d6a05,520,2026-10-10T14:08:50+00:00,bffc04bafca9,~/git/window-ml-cuts2/tests/e2e/artifacts/bench,"[prompt-budget-cuts2, cuts2-recheck2, cuts2-py..."


## Passes per model, current prompt vs round 2

In [3]:
def passes(df):
    t = df.pivot_table(index="model", columns="prompt", values="passed", aggfunc=["sum", "count"])
    out = pd.DataFrame({p: t["sum"][p].astype(int).astype(str) + "/" + t["count"][p].astype(str) for p in ["current", "cuts2"]})
    out["change"] = t["sum"]["cuts2"] - t["sum"]["current"]
    return out.sort_values("change")

passes(runs[runs.sweep == "prompt-budget-cuts2"])

,current,cuts2,change
model,,,
glm-4.7-flash:latest,7/12,5/12,-2
openrouter.minimax/minimax-m3,11/12,9/12,-2
qwen3.6:35b,11/12,10/12,-1
deepseek.deepseek-flash,12/12,12/12,0
gemma4:31b,12/12,12/12,0
litellm.anthropic/claude-sonnet-5-5,12/12,12/12,0
litellm.google/gemini-3.6-flash,12/12,12/12,0
moonshot.kimi-k3,12/12,12/12,0
openrouter.openai/gpt-6-luna,12/12,12/12,0


In [4]:
recheck = runs[runs.sweep == "cuts2-recheck2"]
passes(recheck)

,current,cuts2,change
model,,,
openrouter.minimax/minimax-m3,31/32,27/32,-4
glm-4.7-flash:latest,14/32,13/32,-1
qwen3.6:35b,31/32,31/32,0


Per task, for the recheck. A drop concentrated in one task points at the text that task needs.

In [5]:
recheck.pivot_table(index=["model", "task"], columns="prompt", values="passed", aggfunc="sum").astype(int)

prompt                                        current  cuts2
model                         task                          
glm-4.7-flash:latest          csv-total             4      2
                              icon-heart            2      2
                              spa-rendered          3      2
                              table-north-q3        5      7
openrouter.minimax/minimax-m3 csv-total             8      7
                              icon-heart            7      5
                              spa-rendered          8      8
                              table-north-q3        8      7
qwen3.6:35b                   csv-total             8      7
                              icon-heart            7      8
                              spa-rendered          8      8
                              table-north-q3        8      8

## Is a drop more than noise?

Fisher's exact test on each model's 2x2 table (passes and failures under each prompt), one-sided towards round 2
being worse. Pooled: both sweeps together, for the models in both.

In [6]:
def fisher(df):
    out = []
    for model, g in df.groupby("model"):
        a = g[g.prompt == "current"].passed; b = g[g.prompt == "cuts2"].passed
        table = [[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]]
        out.append({"model": model, "current": f"{a.sum()}/{len(a)}", "cuts2": f"{b.sum()}/{len(b)}",
                    "p (cuts2 worse)": round(fisher_exact(table, alternative="greater").pvalue, 3)})
    return pd.DataFrame(out).set_index("model")

both = runs[runs.sweep.isin(["prompt-budget-cuts2", "cuts2-recheck2"]) & runs.model.isin(recheck.model.unique())]
pd.concat({"recheck": fisher(recheck), "pooled": fisher(both)}, axis=1)

recheck                         pooled                       
                              current  cuts2 p (cuts2 worse) current  cuts2 p (cuts2 worse)
model                                                                                      
glm-4.7-flash:latest            14/32  13/32           0.500   21/44  18/44           0.334
openrouter.minimax/minimax-m3   31/32  27/32           0.098   42/44  36/44           0.045
qwen3.6:35b                     31/32  31/32           0.754   42/44  41/44           0.500

## Why each round-2 failure failed

Classified from the run's own steps, in this order: the step cap; `tables=` written inside the Python code instead of
passed as the argument; a table task answered without `python_exec`; the right status read but clicked through `exec`
(the task's check asks for the `click` tool); otherwise an answer not taken from the page.

In [7]:
TABLE_TASKS = {"csv-total", "table-north-q3"}

def why(r):
    if r.hit_cap: return "step cap"
    if r.tables_in_code: return "tables= written in the code"
    if r.task in TABLE_TASKS and "python_exec" not in r.tools: return "table task without python_exec"
    if r.task == "icon-heart" and "Clicked: heart" in r.answer and "click" not in r.tools: return "clicked via exec (scoring)"
    return "answer not from the page"

failed = recheck[~recheck.passed].copy()
failed["why"] = failed.apply(why, axis=1)
failed.pivot_table(index=["model", "why"], columns="prompt", values="repeat", aggfunc="count", fill_value=0)

prompt                                                        current  cuts2
model                         why                                           
glm-4.7-flash:latest          answer not from the page             10     15
                              step cap                              8      4
openrouter.minimax/minimax-m3 answer not from the page              0      1
                              clicked via exec (scoring)            0      1
                              step cap                              1      2
                              table task without python_exec        0      1
qwen3.6:35b                   answer not from the page              1      0
                              step cap                              0      1

## After one sentence: the table tasks again

`PYTHON_SHORT.tables` gained "Pass it HERE, not in your code: the code then just uses `df` (or each name)." Both
prompts were re-run on the same build, so the comparison is within this sweep only.

In [8]:
fix = runs[runs.sweep.isin(FIX_SWEEPS)]
fix.pivot_table(index=["model", "task"], columns="prompt", values="passed", aggfunc=["sum", "count"])

sum         count      
prompt                                       current cuts2 current cuts2
model                         task                                      
glm-4.7-flash:latest          csv-total            4     2       8     8
                              table-north-q3       5     7       8     8
openrouter.minimax/minimax-m3 csv-total            8     8       8     8
                              table-north-q3       8     8       8     8

## What the cut saves: prompt tokens per model call

In [9]:
r = recheck[recheck.calls > 0].assign(per_call=lambda d: d.prompt_tokens / d.calls)
t = r.pivot_table(index="model", columns="prompt", values="per_call", aggfunc="mean").round(0)
t["saved"] = (1 - t["cuts2"] / t["current"]).map("{:.1%}".format)
t

prompt,current,cuts2,saved
model,,,
glm-4.7-flash:latest,10527.0,9919.0,5.8%
openrouter.minimax/minimax-m3,10675.0,9972.0,6.6%
qwen3.6:35b,11229.0,10501.0,6.5%


## The ship rule

A model fails the rule when it passes fewer runs under round 2 than under the current prompt in the recheck, after
setting aside failures the classification attributes to scoring. For a model re-run after the fix, those runs replace
the recheck's on the table tasks.

In [10]:
def verdict():
    out = []
    for model, g in recheck.groupby("model"):
        g = g.copy()
        g["counted"] = g.passed | g.apply(lambda r: not r.passed and why(r) == "clicked via exec (scoring)", axis=1)
        refit = fix[fix.model == model]
        if len(refit):
            g = pd.concat([g[~g.task.isin(TABLE_TASKS)], refit.assign(counted=refit.passed)])
        cur, cut = g[g.prompt == "current"].counted.sum(), g[g.prompt == "cuts2"].counted.sum()
        a = [[cur, len(g[g.prompt == "current"]) - cur], [cut, len(g[g.prompt == "cuts2"]) - cut]]
        out.append({"model": model, "current": int(cur), "cuts2": int(cut), "re-run after fix": bool(len(refit)),
                    "holds": bool(cut >= cur), "p (cuts2 worse)": round(fisher_exact(a, alternative="greater").pvalue, 3)})
    return pd.DataFrame(out).set_index("model")

verdict()

,current,cuts2,re-run after fix,holds,p (cuts2 worse)
model,,,,,
glm-4.7-flash:latest,14,13,True,False,0.500
openrouter.minimax/minimax-m3,31,30,True,False,0.500
qwen3.6:35b,31,31,False,True,0.754


## Why glm-4.7-flash still lost csv-total after the fix

The same classification over glm's table-task runs, the recheck and the re-run after the fix together. A class that
appears under one prompt only is a behaviour that prompt caused.

In [11]:
glm = runs[runs.sweep.isin(["cuts2-recheck2", *FIX_SWEEPS]) & (runs.model == "glm-4.7-flash:latest") & runs.task.isin(TABLE_TASKS)]
g = glm[~glm.passed].assign(why=lambda d: d.apply(why, axis=1))
passed = glm.pivot_table(index="task", columns="prompt", values="passed", aggfunc=["sum", "count"])
display(passed)
g.pivot_table(index=["task", "why"], columns="prompt", values="repeat", aggfunc="count", fill_value=0)

sum         count      
prompt         current cuts2 current cuts2
task                                      
csv-total            8     4      16    16
table-north-q3      10    14      16    16

prompt                                         current  cuts2
task           why                                           
csv-total      answer not from the page              2      4
               step cap                              6      5
               table task without python_exec        0      3
table-north-q3 answer not from the page              4      2
               step cap                              1      0
               table task without python_exec        1      0

## Decision (2026-10-10)

Shipped: the `locate` cut and the private-rendering error that leads with its retry. Not shipped: the `python_exec`
cut. With the `tables` fix MiniMax recovered on the table tasks, but glm-4.7-flash gave up on csv-total without
calling `python_exec` only under the cut (the table above), so `python_exec` keeps its full texts and cannot have
regressed. The `locate` cut held on icon-heart for qwen3.6 and glm-4.7-flash; MiniMax's remaining gap there is one
run in eight. The record is `docs/spec/PROMPT_BUDGET.md`.